In [0]:

# ==========================================
# 01 - SYNTHETIC DATA GENERATION
# Fraud Detection Lakehouse
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# 1. Project configuration
# ------------------------------------------

CATALOG_NAME = "fraud_detection"

CUSTOMER_COUNT = 10_000
MERCHANT_COUNT = 1_000
TRANSACTION_COUNT = 100_000

# Unity Catalog Volume for raw source files
VOLUME_NAME = "raw_data"

RAW_PATH = f"/Volumes/{CATALOG_NAME}/default/{VOLUME_NAME}"

# Reproducibility
RANDOM_SEED = 42

# ------------------------------------------
# 2. Create raw data volume
# ------------------------------------------

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOG_NAME}.default.{VOLUME_NAME}
""")

# ------------------------------------------
# 3. Generate customers
# ------------------------------------------

customers = (
    spark.range(1, CUSTOMER_COUNT + 1)
    .select(
        F.concat(
            F.lit("CUST-"),
            F.lpad(F.col("id").cast("string"), 6, "0")
        ).alias("customer_id"),

        (F.floor(F.rand(11) * 63) + 18)
        .cast("int").alias("age"),

        F.element_at(
            F.array(
                F.lit("PH"),
                F.lit("US"),
                F.lit("SG"),
                F.lit("AU"),
                F.lit("GB")
            ),
            (F.floor(F.rand(12) * 5) + 1).cast("int")
        ).alias("country"),

        F.date_sub(
            F.current_date(),
            (F.floor(F.rand(13) * 1825)).cast("int")
        ).alias("registration_date")
    )
)

# ------------------------------------------
# 4. Generate merchants
# ------------------------------------------

merchants = (
    spark.range(1, MERCHANT_COUNT + 1)
    .select(
        F.concat(
            F.lit("MER-"),
            F.lpad(F.col("id").cast("string"), 5, "0")
        ).alias("merchant_id"),

        F.element_at(
            F.array(
                F.lit("Retail"),
                F.lit("Grocery"),
                F.lit("Travel"),
                F.lit("Electronics"),
                F.lit("Food"),
                F.lit("Entertainment"),
                F.lit("Healthcare"),
                F.lit("Online Services")
            ),
            (F.floor(F.rand(21) * 8) + 1).cast("int")
        ).alias("category"),

        F.element_at(
            F.array(
                F.lit("PH"),
                F.lit("US"),
                F.lit("SG"),
                F.lit("AU"),
                F.lit("GB")
            ),
            (F.floor(F.rand(22) * 5) + 1).cast("int")
        ).alias("country")
    )
)

# ------------------------------------------
# 5. Generate transactions
# ------------------------------------------

transactions_base = spark.range(0, TRANSACTION_COUNT)

transactions = (
    transactions_base
    .select(
        F.concat(
            F.lit("TXN-"),
            F.lpad(
                (F.col("id") + 1).cast("string"),
                8,
                "0"
            )
        ).alias("transaction_id"),

        F.concat(
            F.lit("CUST-"),
            F.lpad(
                (
                    F.pmod(
                        F.xxhash64("id"),
                        F.lit(CUSTOMER_COUNT)
                    ) + 1
                ).cast("string"),
                6,
                "0"
            )
        ).alias("customer_id"),

        F.concat(
            F.lit("MER-"),
            F.lpad(
                (
                    F.pmod(
                        F.xxhash64("id"),
                        F.lit(MERCHANT_COUNT)
                    ) + 1
                ).cast("string"),
                5,
                "0"
            )
        ).alias("merchant_id"),

        F.round(
            F.rand(31) * 99995 + 5,
            2
        ).alias("amount"),

        F.from_unixtime(
            F.unix_timestamp(
                F.lit("2026-11-16 00:00:00")
            ) - (F.rand(32) * 31536000).cast("long")
        ).cast("timestamp").alias("transaction_timestamp"),

        F.element_at(
            F.array(
                F.lit("Online"),
                F.lit("POS"),
                F.lit("Mobile")
            ),
            (F.floor(F.rand(33) * 3) + 1).cast("int")
        ).alias("channel"),

        F.element_at(
            F.array(
                F.lit("Credit Card"),
                F.lit("Debit Card"),
                F.lit("Bank Transfer"),
                F.lit("Digital Wallet")
            ),
            (F.floor(F.rand(34) * 4) + 1).cast("int")
        ).alias("payment_method"),

        F.concat(
            F.lit("DEV-"),
            F.lpad(
                (
                    F.pmod(
                        F.xxhash64("id"),
                        F.lit(20000)
                    ) + 1
                ).cast("string"),
                5,
                "0"
            )
        ).alias("device_id"),

        F.element_at(
            F.array(
                F.lit("PH"),
                F.lit("US"),
                F.lit("SG"),
                F.lit("AU"),
                F.lit("GB")
            ),
            (F.floor(F.rand(35) * 5) + 1).cast("int")
        ).alias("country"),

        F.when(
            F.rand(36) < 0.97,
            F.lit("COMPLETED")
        ).otherwise(F.lit("DECLINED")).alias("status")
    )
)

# ------------------------------------------
# 6. Generate synthetic fraud labels
# ------------------------------------------

transactions = transactions.withColumn(
    "fraud_label",
    F.when(
        (F.col("amount") >= 70000) & (F.rand(42) > 0.60),
        1
    ).when(
        (F.col("amount") >= 30000) & (F.rand(43) > 0.985),
        1
    ).otherwise(0)
)

# ------------------------------------------
# 7. Save source datasets as CSV
# ------------------------------------------

customers.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{RAW_PATH}/customers")

merchants.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{RAW_PATH}/merchants")

transactions.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{RAW_PATH}/transactions")

print("Synthetic data generation completed!")


In [0]:
# Verify record counts

datasets = {
    "Customers": f"{RAW_PATH}/customers",
    "Merchants": f"{RAW_PATH}/merchants",
    "Transactions": f"{RAW_PATH}/transactions"
}

for name, path in datasets.items():
    df = spark.read.option("header", True).csv(path)
    print(f"{name}: {df.count():,} records")

In [0]:
# Preview transactions

display(
    spark.read
    .option("header", True)
    .csv(f"{RAW_PATH}/transactions")
    .limit(20)
)